# Day 75: Model Deployment Basics

## Saving a Trained Model
Training a model can take minutes, hours, or days, so you don't want to retrain it every time you need to make a prediction - instead you train once and save ("serialize") the trained model's learned parameters to a file, then load that file whenever you need it. `joblib` is the standard tool for this in the scikit-learn ecosystem (it's more efficient than plain `pickle` for objects containing large numpy arrays, which is exactly what a trained model like a RandomForest is full of). The code below trains a RandomForestClassifier on the classic Iris flower dataset and writes it to `iris_model.pkl` - this `.pkl` file is now a portable, reusable artifact that captures everything the model learned, ready to be loaded elsewhere (a different script, a web server, another machine) without needing the original training code or data again.</cell id="cell-1">


In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import joblib

X, y = load_iris(return_X_y=True)
model = RandomForestClassifier(random_state=42).fit(X, y)  # train once; random_state makes results reproducible
joblib.dump(model, 'iris_model.pkl')  # serialize the trained model (weights/trees + config) to disk
print('Model saved to iris_model.pkl')

## Loading and Reusing the Model
This is the other half of the save/load pattern: `joblib.load` reads the `.pkl` file back into memory as a fully-trained model object, identical to the one we just trained - no retraining needed. In a real deployment, this loading typically happens once when a service starts up (not on every request, which would be slow), and the resulting model object is then reused to answer many prediction requests. Calling `.predict` on a new set of feature values (here, one iris flower's four measurements: sepal length, sepal width, petal length, petal width) returns the predicted class - this is exactly the kind of call a web API or application would make behind the scenes to serve a prediction to a user.</cell id="cell-3">


In [ ]:
loaded_model = joblib.load('iris_model.pkl')  # loads the fully-trained model back into memory, ready for predictions
print('Prediction:', loaded_model.predict([[5.1, 3.5, 1.4, 0.2]]))  # one sample: [sepal_length, sepal_width, petal_length, petal_width]

## Serving via a Flask API
Saving a model to a file is only useful if other systems (a website, a mobile app, another service) can actually get predictions from it - the most common way to expose a model is behind a web API that other programs call over HTTP. Flask is a lightweight Python web framework well suited for this: the code below defines a single `/predict` endpoint that accepts a POST request with JSON containing a `features` list, runs those features through the model we saved earlier, and returns the predicted class as a JSON response. Because the model is loaded once at startup (not inside the `predict` function), every incoming request reuses the same already-loaded model, which keeps response times fast. This pattern - a thin web layer wrapped around a serialized model - is the backbone of most real-world "model as a service" deployments, even though production systems typically add things like input validation, authentication, and a production-grade server instead of Flask's built-in dev server.</cell id="cell-5">


In [ ]:
# pip install flask
# app.py
from flask import Flask, request, jsonify
import joblib

app = Flask(__name__)
model = joblib.load('iris_model.pkl')  # load the model once at startup, not per-request, so predictions stay fast

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()  # expects JSON body like {"features": [5.1, 3.5, 1.4, 0.2]}
    prediction = model.predict([data['features']])  # wrap in a list since predict expects a batch of samples
    return jsonify({'prediction': int(prediction[0])})  # int() converts numpy int type to a plain JSON-serializable int

if __name__ == '__main__':
    app.run(debug=True)  # debug=True is convenient for development only - never use it in production
# Run with: python app.py, then POST to http://localhost:5000/predict

## Serving via Streamlit
A Flask API is great for programmatic access, but it has no visual interface - anyone using it needs to send raw HTTP requests. Streamlit takes a different approach aimed at quickly building interactive tools: with just a few lines of pure Python, it turns your script into a browser-based UI with widgets like sliders, buttons, and text inputs, no HTML/CSS/JavaScript required. This is especially useful for internal tools - letting a non-technical teammate or stakeholder try a model interactively (adjust a slider, click predict, see the result) without needing to know how to call an API. The trade-off is that Streamlit apps are generally meant for human-in-the-loop, exploratory, or internal use rather than high-throughput automated integrations, which is where a Flask/FastAPI-style API is still the better choice.</cell id="cell-7">


In [ ]:
# pip install streamlit
# app.py
import streamlit as st
import joblib

model = joblib.load('iris_model.pkl')  # load the saved model once when the app starts
st.title('Iris Species Predictor')
sepal_length = st.slider('Sepal Length', 4.0, 8.0, 5.1)  # interactive slider widget; 5.1 is the default value
if st.button('Predict'):  # only run prediction when the user clicks the button
    st.write('Prediction:', model.predict([[sepal_length, 3.5, 1.4, 0.2]])[0])  # other 3 features left fixed for this demo
# Run with: streamlit run app.py